In [ ]:
# PyTorch Compile Analysis Part 3: Graph Capture and Inspection
# Question 2: make_fx Intermediate Representation

import torch
import torch.nn.functional as nnf ## Only needed if symbolic tracing is being done.
import torch._dynamo as dyn
from torch.fx import symbolic_trace ## Only needed if symbolic tracing is being done.

layerNormalization = nn.LayerNorm(28).to('cuda')

t1 = torch.rand(1024, 28, 28, device='cuda')
t2 = torch.rand(1024, 28, 28, device='cuda')
x = torch.rand(1024, 28, 28, device='cuda')

def doOperations(x, list, t1, t2):
    results = x @ t1
    results = results @ t2
    results = torch.relu(results)
    results = results + x
    results = nnf.layer_norm(results, normalized_shape=28) # Needed if symbolic tracing is being done

    print("We have performed 5 operations so far")
    list.append(42)

    return results

tracedFunction = symbolic_trace(doOperations)
print(tracedFunction)


In [19]:
# PyTorch Compile Analysis Part 3: Graph Capture and Inspection
# Question 2: Standard Execution (without make_fx Intermediate Representation)

import torch
import torch.nn as nn
import torch._dynamo as dyn

layerNormalization = nn.LayerNorm(28).to('cuda')

t1 = torch.rand(1024, 28, 28, device='cuda')
t2 = torch.rand(1024, 28, 28, device='cuda')
x = torch.rand(1024, 28, 28, device='cuda')

def doOperations(x, list, t1, t2):
    results = x @ t1
    results = results @ t2
    results = torch.relu(results)
    results = results + x
    results = layerNormalization(results)

    print("We have performed 5 operations so far")
    print("Before list append: ", list)
    list.append(42)
    print("After list append: ", list)

    return results

list1 = []
compiledFunction = torch.compile(doOperations, fullgraph=True)
dyn.explain(compiledFunction)(x, list1.copy(), t1, t2) # Uses copy of list1 to prevent duplicate insertions for display purposes


We have performed 5 operations so far
Before list append:  []
After list append:  [42]


ExplainOutput(graphs=[GraphModule()], graph_count=1, graph_break_count=0, break_reasons=[GraphCompileReason(reason="Failed to trace builtin operator\n  Explanation: Dynamo does not know how to trace builtin operator `print` with argument types ['str'] (has_kwargs False)\n  Hint: Avoid calling builtin `print` with argument types ['str']. Consider using an equivalent alternative function/method to `print`.\n  Hint: If you are attempting to call a logging function (e.g. `print`), you can try adding it to `torch._dynamo.config.reorderable_logging_functions`.\n  Hint: Please report an issue to PyTorch.\n\n  Developer debug context: builtin print [<class 'torch._dynamo.variables.constant.ConstantVariable'>] False\n\n For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0059.html", user_stack=[<FrameSummary file /tmp/ipykernel_4990/3757311413.py, line 21 in doOperations>], graph_break=True)], op_count=5, ops_per_graph=[[<built-in 

In [20]:
# PyTorch Compile Analysis Part 3: Graph Capture and Inspection
# Question 1

import torch
import torch.nn as nn

layerNormalization = nn.LayerNorm(1024)

def doOperations(x, list):
    results = x @ torch.randn(1024, 1024)
    results = results @ torch.randn(1024, 1024)
    results = torch.relu(results)
    results = results + x
    results = layerNormalization(results)

    print("We have performed 5 operations so far")
    list.append(42)

    return results

x = torch.rand(1024, 1024)
list = [1,2,3,4]

print("Before DoOperation: ", list)
output = doOperations(x, list)

print("After DoOperation: ", list)

print("Output tensor:")
print(output)

Before DoOperation:  [1, 2, 3, 4]
We have performed 5 operations so far
After DoOperation:  [1, 2, 3, 4, 42]
Output tensor:
tensor([[-6.8513e-01, -6.8454e-01,  6.9197e-01,  ..., -6.8508e-01,
          1.2050e+00, -3.1482e-01],
        [-6.6379e-01, -6.6396e-01,  2.8785e+00,  ..., -6.6397e-01,
         -6.6428e-01, -6.6360e-01],
        [-6.9009e-01, -6.8972e-01,  2.2248e+00,  ..., -6.8731e-01,
          1.3355e+00, -6.9003e-01],
        ...,
        [-6.8578e-01, -6.8401e-01,  1.2367e+00,  ..., -6.8607e-01,
          1.8238e+00, -2.1056e-01],
        [-6.9836e-01, -6.9852e-01,  1.7545e+00,  ..., -6.9794e-01,
          1.3335e-03, -4.1068e-01],
        [-6.8859e-01, -6.8806e-01,  2.4354e+00,  ..., -6.8841e-01,
          1.2972e-01, -3.8143e-01]], grad_fn=<NativeLayerNormBackward0>)


In [21]:
# PyTorch Compile Analysis Part 2

import torch
import torch._dynamo as dyn

def problem3(x):
    # This might have suboptimal performance
    # result = 0
    # for i in range (10) :
    #     result += (x ** i).sum()
    # return result
    # Solution
    total = torch.stack([x ** i for i in range(10)])
    return total

compiledFunction = torch.compile(problem2)
x = torch.rand(1024, 28, 28, device='cuda')
dyn.explain(problem2)(x)

NameError: name 'problem2' is not defined

In [ ]:
# PyTorch Compile Analysis Part 2

mport torch
import torch._dynamo as dyn

def problem2(x):
    # This should also have issues
    # d = {}
    # d['key'] = x
    # return d['key'] * 2
    # Solution
    return x*2

compiledFunction = torch.compile(problem2)
x = torch.rand(1024, 28, 28, device='cuda')
dyn.explain(problem2)(x)


In [ ]:
# PyTorch Compile Analysis Part 2

import torch
import torch._dynamo as dyn

def problem1(x):
    # This should fail to compile
    # if x.sum() > 0:
    #     return x * 2
    # else:
    #     return x / 2
    # Solution
    return torch.where(x.sum()>0, x*2, x/2)

compiledFunction = torch.compile(problem1)
x = torch.rand(1024, 28, 28, device='cuda')
# compiledFunction(x)

dyn.explain(problem1)(x)

In [ ]:
# PyTorch Compile Analysis Part 1
# 100 Forward + Backward Passes
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time
import matplotlib.pyplot as plt

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 10),
            nn.ReLU(),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
torch.set_float32_matmul_precision('high') # Changed to high for better performance, set to medium by default

backends = [
    "eager",      # No compilation
    "aot_eager",  # AOTAutograd without Inductor
    "inductor",   # Default Inductor backend
    "cudagraphs", # CUDA graphs
]

results = {}

model = NeuralNetwork().to('cuda')
# print(model)
x = torch.rand(1024, 28, 28, device='cuda')

# fake label for testing
batchSize = 1024
numClasses = 10
fakeLabel = torch.randint(0, numClasses, (batchSize,), device='cuda')

lossFunction = nn.CrossEntropyLoss()

for backend in backends:
    compiledModel = torch.compile(model, backend=backend)
    # warmup
    for _ in range(3):
        output = compiledModel(x)
        loss = lossFunction(output, fakeLabel)
        loss.backward()
        compiledModel.zero_grad(set_to_none=True)

    torch.cuda.synchronize()
    start = time.time()
    for _ in range(100):
        compiledModel.zero_grad(set_to_none=True)
        output = compiledModel(x)
        loss = lossFunction(output, fakeLabel)
        loss.backward()

    torch.cuda.synchronize()
    end = time.time()
    avgTime = (end - start) / 100
    results[backend] = avgTime
    print(f"Backend: {backend}, Avg time per forward+backward pass: {end-start:.12f} seconds")

plt.figure(figsize=(8,5))
plt.bar(results.keys(), results.values(), color=['blue', 'orange', 'red', 'green'])
plt.ylabel("Avg time per forward+backpass pass (in seconds)")
plt.ylim(0, max(results.values())*1.2) # Add space above tallest bar in case y limit is too small
plt.xlabel("PyTorch Backends")
plt.grid(axis='y', linestyle='--', alpha=0.5)

for i, backend in enumerate(results.keys()):
    plt.text(i, results[backend] + 0.00005, f"{results[backend]:.6f}", ha='center', va='bottom')

plt.show()

In [ ]:
# PyTorch Compile Analysis
# 3 Layer NN
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 10),
            nn.ReLU(),
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
torch.set_float32_matmul_precision('high') # Changed to high for better performance, set to medium by default

backends = [
    "eager",      # No compilation
    "aot_eager",  # AOTAutograd without Inductor
    "inductor",   # Default Inductor backend
    "cudagraphs", # CUDA graphs
]

results = {}

model = NeuralNetwork().to('cuda')
# print(model)
x = torch.rand(1024, 28, 28, device='cuda')

for backend in backends:
    compiledModel = torch.compile(model, backend=backend)
    for _ in range(3):
        _ = compiledModel(x)
        

    start = time.time()
    output = compiledModel(x)
    end = time.time()

    results[backend] = {
        "output": output,
        "time in seconds": end-start,
    }
    print(f"Backend: {backend}, Time: {end-start:.12f} sec")
